# Visualize correlations between Donor Genomics and Metadata: REDS Index
## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from collections import defaultdict
from itertools import product
import re
import textwrap

import matplotlib as mpl
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
import numpy as np
import pandas as pd
import seaborn as sns

from hk1_r12ter_analysis.config import (
    GENE_ID,
    INTERIM_DATA_DIR,
    PROCESSED_DATA_DIR,
    PROJ_ROOT,
    RAW_DATA_DIR,
    REPORTS_DIR,
    logger,
)
from hk1_r12ter_analysis.io import load_dataframe_from_file, make_filename
from hk1_r12ter_analysis.linkage_disequilibrium import identify_independent_snps
from hk1_r12ter_analysis.util import ceil_to_decimal
from hk1_r12ter_analysis.visualize import (
    plot_correlations_vs_pvalues,
    plot_violin_for_snp_phenotype,
)

## REDS Index

In [ ]:
sample_key = "INDEX DONOR ID"
group_key = None
source_type = "REDS-Index"
main_data_type = f"Genomics-{GENE_ID}"
other_data_type = "Metadata"
norm_str = "median-none-auto"

correlation_statistic = "Spearman"
r2_threshold = 0.8
prioritize_genotyped_in_LD = True

filetype = "pdf"
variable_fancy_replacements = {
    f"{source_type.split('-')[-1]}.Adjusted.Osmotic.Hemolysis": "Osmotic Hemolysis",
    f"{source_type.split('-')[-1]}.Adjusted.Storage.Hemolysis": "Storage Hemolysis",
    f"{source_type.split('-')[-1]}.Adjusted.Oxidative.Hemolysis": "Oxidative Hemolysis",
    source_type: source_type.replace("-", " "),
}

### Load correlation and statistics

In [ ]:
input_dirpath = PROCESSED_DATA_DIR / "REDS" / "Correlations"
input_filename = make_filename(source_type, f"{main_data_type}_All_Data", "Statistics")
df_correlations_tall = load_dataframe_from_file(
    input_dirpath / input_filename, index_col=None, low_memory=False
)

group_value = "ALL"
correlations_options = {
    "Group": group_value,
    "DataType1": main_data_type,
    "DataType2": other_data_type,
}
for key, value in correlations_options.items():
    df_correlations_tall = (
        df_correlations_tall[df_correlations_tall[key] == value]
        .drop_duplicates()
        .drop(key, axis=1)
    )
df_correlations_tall = df_correlations_tall.set_index(["Variable2", "Variable1"])
df_correlations_tall.index.names = ["Variable", "ID"]
df_correlations_tall

### Load data
#### Load numerical metadata and genomics

In [ ]:
# Load metadata + genomics
input_dirpath = PROCESSED_DATA_DIR / "REDS" / "Normalized" / norm_str
input_filename = make_filename(source_type, "Donor", "RBC", "All_Data")
df_all_data = load_dataframe_from_file(
    input_dirpath / input_filename,
    index_col=[key for key in [sample_key, group_key] if key],
    header=[0, 1],
)
df_all_data = df_all_data.convert_dtypes()

df_data = df_all_data.loc[:, [main_data_type, other_data_type]]
if group_key:
    if group_value != "ALL":
        df_data = df_data.loc[pd.IndexSlice[:, group_value], :]
    df_data = df_data.droplevel(group_key).drop_duplicates()
df_data

### Load genomics metadata

In [ ]:
input_dirpath = PROCESSED_DATA_DIR / "REDS"
input_filename = make_filename("REDS", "Genomics", GENE_ID, "Metadata")
df_genomics_ann = load_dataframe_from_file(input_dirpath / input_filename, index_col="ID")
df_genomics_ann = df_genomics_ann.loc[df_all_data.loc[:, f"Genomics-{GENE_ID}"].columns.tolist()]
df_genomics_ann = df_genomics_ann.drop("ANN_Priority", axis=1)
df_rsid_map = df_genomics_ann["RSID"].replace(":", ".")
df_genomics_ann

#### Load linkage disequilibrium $r^{2}$ correlation matrices

In [ ]:
LD_method = "RH"
rsids = df_genomics_ann.index.tolist()
input_dirpath = PROCESSED_DATA_DIR / "REDS" / "Correlations"
input_filename = make_filename(
    "REDS-Index",  # Always REDS-Index cohort
    f"Genomics-{GENE_ID}_Metadata",
    LD_method,
    "LinkageDisequilibrium",
    "ALL",
)
df_ld_matrix = load_dataframe_from_file(input_dirpath / input_filename, index_col=["SNP1", "SNP2"])
df_r2_matrix = df_ld_matrix.loc[:, "r2"]
df_r2_matrix

## Visualize metadata values based on phenotype for given SNP

In [ ]:
variable = f"{source_type.split('-')[-1]}.Adjusted.Osmotic.Hemolysis"
variable_fancy = variable_fancy_replacements.get(variable, variable)
df = df_correlations_tall.loc[variable]
df = df.sort_values(by=f"{correlation_statistic} -log10(p-value)", ascending=False)
df = df.join(df_genomics_ann, how="left")
cols_of_interest = [
    "RSID",
    "Spearman statistic",
    "Spearman -log10(p-value)",
    "ANN_Annotation_Impact",
    "ANN_Gene_Name",
    "ANN_Feature_Type",
    "ANN_Transcript_BioType",
    "ANN_HGVS.c",
    "ANN_HGVS.p",
]
if prioritize_genotyped_in_LD:
    df.loc[:, "Type"] = df.reset_index(drop=False)["ID"].str.split("_", n=1, expand=True)[0].values
    df = df.sort_values(
        by=["Type", f"{correlation_statistic} -log10(p-value)"], ascending=[True, False]
    ).drop("Type", axis=1)
else:
    df = df.sort_values(by=f"{correlation_statistic} -log10(p-value)", ascending=False)
rsids = identify_independent_snps(
    df_r2_matrix=df_r2_matrix,
    list_of_snps=df.index.get_level_values("ID").tolist(),
    r2_threshold=r2_threshold,
)
df = df.loc[rsids, cols_of_interest].sort_values(
    by=f"{correlation_statistic} -log10(p-value)", ascending=False
)
rsids = df.index.tolist()
df

In [ ]:
# Get data to plot
data = df_correlations_tall.copy()
data = data.loc[
    pd.IndexSlice[variable, rsids],
    [f"{correlation_statistic} statistic", f"{correlation_statistic} -log10(p-value)"],
]
data = data.sort_values(by=f"{correlation_statistic} -log10(p-value)", ascending=False)

group_options = {
    "ALL": {
        "Osmotic Hemolysis": {
            "xdecimals": 2,
            "xscalar": 4,
            "xmajor_tick_multiple": 0.1,
            "ydecimals": 0,
            "yscalar": 20,
            "ymajor_tick_multiple": 40,
        },
        "Oxidative Hemolysis": {
            "xdecimals": 2,
            "xscalar": 1,
            "xmajor_tick_multiple": 0.05,
            "ydecimals": 1,
            "yscalar": 5,
            "ymajor_tick_multiple": 1,
        },
        "Storage Hemolysis": {
            "xdecimals": 3,
            "xscalar": 1,
            "xmajor_tick_multiple": 0.05,
            "ydecimals": 1,
            "yscalar": 5,
            "ymajor_tick_multiple": 1,
        },
    },
}

group_value_kwargs = group_options[group_value][variable_fancy]
xlim_pads = (0.0, 0.125)
ylim_pads = (0.025, 0.0)

title = "\n".join(
    [
        f"{GENE_ID} SNP Correlates",
        f"with {other_data_type} in",
        f"{variable_fancy_replacements.get(source_type)} Donors"
        + (f", {group_key} {group_value}" if group_value != "ALL" else ""),
    ]
)

xabsmax = ceil_to_decimal(
    data[f"{correlation_statistic} statistic"].abs().max(),
    decimals=group_value_kwargs["xdecimals"],
    scalar=group_value_kwargs["xscalar"],
)
xlim = (-xabsmax * (1 + xlim_pads[0]), xabsmax * (1 + xlim_pads[1]))
print(f"x-limits: {xlim[0]:.4f}, {xlim[1]:.4f}")
yabsmax = ceil_to_decimal(
    data[f"{correlation_statistic} -log10(p-value)"].abs().max(),
    decimals=group_value_kwargs["ydecimals"],
    scalar=group_value_kwargs["yscalar"],
)
ylim = (-yabsmax * ylim_pads[0], yabsmax * (1 + ylim_pads[1]))
print(f"y-limits: {ylim[0]:.4f}, {ylim[1]:.4f}")

# Generate figure
fig, ax = plt.subplots(figsize=(4, 4), nrows=1, ncols=1)
cax, ax = (None, ax) if isinstance(ax, mpl.axes.Axes) else ax
ax, cax = plot_correlations_vs_pvalues(
    data=data,
    correlation_statistic=data.columns[0],
    neg_log10_pvalue=data.columns[1],
    ax=ax,
    s=40,
    fontdict=dict(size="large"),
    title=title,
    xaxis_kwargs=dict(
        lim=xlim,
        label=f"Spearman correlation ($\\rho$)",
        major_tick_multiple=group_value_kwargs["xmajor_tick_multiple"],
        minor_ticks=True,
        tick_labelsize="medium",
    ),
    yaxis_kwargs=dict(
        lim=ylim,
        label="-log$_{10}$($p$-value)",
        major_tick_multiple=group_value_kwargs["ymajor_tick_multiple"],
        minor_ticks=True,
        tick_labelsize="medium",
    ),
    grid_kwargs=dict(
        linestyle="-",
        color="xkcd:light grey",
        which="both",
    ),
    cax=cax,
)


sns.despine(ax=ax)
fig.tight_layout()

output_dirpath = (
    REPORTS_DIR
    / "REDS"
    / "Correlations"
    / main_data_type
    / other_data_type
    / f"{source_type.split('-')[-1]}.{variable_fancy.replace(' ', '.')}"
)
output_dirpath.mkdir(parents=True, exist_ok=True)
filename_args = [source_type, f"{correlation_statistic}Correlations"]
if group_value != "ALL":
    filename_args += [f"{group_key}{group_value}"]
if prioritize_genotyped_in_LD:
    filename_args += ["g_priority"]
output_filepath = output_dirpath / make_filename(*filename_args, filetype=filetype)
fig.savefig(output_filepath)
logger.info(f"Saved '{filetype}' file successfully to '{output_filepath.relative_to(PROJ_ROOT)}'.")
plt.show()
plt.close()
data.sort_values(by=f"{correlation_statistic} -log10(p-value)", ascending=False).head(10)

### Violin + boxplots for SNP subgroups

In [ ]:
unit = "%"

linewidth = 1.5
fontdict = {"size": "large"}
violinplot_kwargs = dict(linewidth=linewidth, bw_method="silverman", bw_adjust=0.5, cut=0)
stripplot_kwargs = dict(s=2, seed=4, color="xkcd:dark grey")
boxplot_kwargs = dict(
    width=0.21,
    capwidth=0.21,
    capprops=dict(linewidth=linewidth * 1.5),
    whiskerprops=dict(linewidth=linewidth * 1.5),
    linewidth=linewidth,
    color="xkcd:light grey",
)

legend_kwargs = dict()
legend_labels = {0: "Homozygous Reference", 1: "Heterozygous", 2: "Homozygous Alternate"}

ylabel = f"{variable_fancy} ({unit})"
ylim_pads = (0.025, 0)
ylim_default = {
    "Osmotic Hemolysis": (0, 100),
    "Oxidative Hemolysis": (0, 100),
    "Storage Hemolysis": (0, 4),
}


ylims_dict = {
    "ALL": {
        "Osmotic Hemolysis": (0, 100),
        "Oxidative Hemolysis": (0, 100),
        "Storage Hemolysis": (0, 4),
    },
}

ylim = ylims_dict.get(group_value, ylim_default)[variable_fancy]
ylim = (-max(ylim) * ylim_pads[0], max(ylim) * (1 + ylim_pads[1]))


output_dirpath = (
    REPORTS_DIR
    / "REDS"
    / "Correlations"
    / main_data_type
    / other_data_type
    / f"{source_type.split('-')[-1]}.{variable_fancy.replace(' ', '.')}"
)
output_dirpath.mkdir(parents=True, exist_ok=True)

df = df_correlations_tall.loc[pd.IndexSlice[variable, rsids], :][
    "Spearman -log10(p-value)"
].sort_values(ascending=False)
rsids_sorted = df.index.get_level_values("ID").tolist()
df

In [ ]:
subgroup = None
split = False
n_rsids = 4
pointplot_kwargs = dict(markersize=5 / (1 if not subgroup else 2), c="xkcd:black")
palette = {0.0: "xkcd:pale blue", 1.0: "xkcd:light blue", 2.0: "xkcd:lightish blue"}

for idx, rsid in enumerate(rsids_sorted):
    xlabel = f"{GENE_ID} {rsid}"
    columns = [(main_data_type, rsid), (other_data_type, variable)]
    sort_by = [rsid]
    if subgroup is not None:
        columns += [("Metadata", subgroup)]
        sort_by += [subgroup]
    data = df_data.loc[:, columns].droplevel(level=0, axis=1).dropna()
    data = data.convert_dtypes().sort_values(by=sort_by)
    stripplot_kwargs["palette"] = {x: "xkcd:grey" for x in data[rsid].unique()}

    # Create figure and plot
    fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(6, 4))
    ax = plot_violin_for_snp_phenotype(
        data=data,
        rsid=rsid,
        variable=variable,
        subgroup=subgroup,
        split=False,
        ax=ax,
        **dict(
            palette=palette,
            violinplot_kwargs=violinplot_kwargs,
            stripplot_kwargs=stripplot_kwargs,
            boxplot_kwargs=boxplot_kwargs,
            pointplot_kwargs=pointplot_kwargs,
            legend_kwargs=legend_kwargs,
        ),
    )

    ax.grid(True)
    ax.set_xlabel(xlabel, fontdict=fontdict)
    ax.set_ylabel(ylabel, fontdict=fontdict)
    if ax.legend_ and not bool(subgroup):
        ax.legend(legend_labels)

    ax.set_xticks(
        ax.get_xticks(),
        labels=[
            legend_labels.get(float(x.get_text()), x.get_text()).replace(" ", "\n")
            for x in ax.get_xticklabels()
        ],
    )
    ax.set_ylim(ylim)
    ax.xaxis.set_tick_params(labelsize="medium")
    ax.yaxis.set_tick_params(labelsize="medium")
    sns.despine(fig)

    filename_args = [source_type, GENE_ID, df_rsid_map.get(rsid, rsid.replace(":", "."))]
    if subgroup is not None:
        filename_args += [subgroup] + ["split" if split else "seperate"]
    if group_value != "ALL":
        filename_args += [f"{group_key}{group_value}"]
    output_filepath = output_dirpath / make_filename(*filename_args, filetype=filetype)
    fig.savefig(output_filepath)
    logger.info(
        f"Saved '{filetype}' file successfully to '{output_filepath.relative_to(PROJ_ROOT)}'."
    )
    plt.show()
    plt.close()
    summary = pd.concat(
        (
            df_correlations_tall.loc[(variable, rsid)].drop(
                ["Spearman p-value", "Group size (0)", "Group size (1)", "Group size (2)"]
            ),
            data.loc[:, sort_by].value_counts(),
        )
    )
    summary[summary.index.str.contains(" p-value").fillna(False)] = summary[
        summary.index.str.contains(" p-value").fillna(False)
    ].apply(lambda x: "{:.3e}".format(x))
    print(rsid)
    print(summary)
    if n_rsids <= idx:
        break

In [ ]:
subgroup = None
split = False
rsid = rsids_sorted[0]
pointplot_kwargs = dict(markersize=5 / (1 if not subgroup else 2), c="xkcd:black")
palette = {0.0: "xkcd:pale blue", 1.0: "xkcd:light blue", 2.0: "xkcd:lightish blue"}

xlabel = f"{GENE_ID} {rsid}"
columns = [(main_data_type, rsid), (other_data_type, variable)]
sort_by = [rsid]
if subgroup is not None:
    columns += [("Metadata", subgroup)]
    sort_by += [subgroup]
data = df_data.loc[:, columns].droplevel(level=0, axis=1).dropna()
data = data.convert_dtypes().sort_values(by=sort_by)
stripplot_kwargs["palette"] = {x: "xkcd:grey" for x in data[rsid].unique()}

# Create figure and plot
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(6, 4))
ax = plot_violin_for_snp_phenotype(
    data=data,
    rsid=rsid,
    variable=variable,
    subgroup=subgroup,
    split=False,
    ax=ax,
    **dict(
        palette=palette,
        violinplot_kwargs=violinplot_kwargs,
        stripplot_kwargs=stripplot_kwargs,
        boxplot_kwargs=boxplot_kwargs,
        pointplot_kwargs=pointplot_kwargs,
        legend_kwargs=legend_kwargs,
    ),
)

ax.grid(True)
ax.set_xlabel(xlabel, fontdict=fontdict)
ax.set_ylabel(ylabel, fontdict=fontdict)
if ax.legend_ and not bool(subgroup):
    ax.legend(legend_labels)

ax.set_xticks(
    ax.get_xticks(),
    labels=[
        legend_labels.get(float(x.get_text()), x.get_text()).replace(" ", "\n")
        for x in ax.get_xticklabels()
    ],
)
ax.set_ylim(ylim)
ax.xaxis.set_tick_params(labelsize="medium")
ax.yaxis.set_tick_params(labelsize="medium")
sns.despine(fig)

filename_args = [source_type, GENE_ID, df_rsid_map.get(rsid, rsid.replace(":", "."))]
if subgroup is not None:
    filename_args += [subgroup] + ["split" if split else "seperate"]
if group_value != "ALL":
    filename_args += [f"{group_key}{group_value}"]
output_filepath = output_dirpath / make_filename(*filename_args, filetype=filetype)
fig.savefig(output_filepath)
logger.info(f"Saved '{filetype}' file successfully to '{output_filepath.relative_to(PROJ_ROOT)}'.")
plt.show()
plt.close()
summary = pd.concat(
    (
        df_correlations_tall.loc[(variable, rsid)].drop(
            ["Spearman p-value", "Group size (0)", "Group size (1)", "Group size (2)"]
        ),
        data.loc[:, sort_by].value_counts(),
    )
)
summary[summary.index.str.contains(" p-value").fillna(False)] = summary[
    summary.index.str.contains(" p-value").fillna(False)
].apply(lambda x: "{:.3e}".format(x))
print(rsid)
print(summary)

### Break down by subgroups
#### Sex

In [ ]:
subgroup = "Sex"
split = True
n_rsids = 4
pointplot_kwargs = dict(markersize=5 / (1 if not subgroup else 2), c="xkcd:black")
palette = {
    (0.0, "M"): "xkcd:pale blue",
    (1.0, "M"): "xkcd:light blue",
    (2.0, "M"): "xkcd:lightish blue",
    (0.0, "F"): "xkcd:pale green",
    (1.0, "F"): "xkcd:light green",
    (2.0, "F"): "xkcd:lightish green",
}

stripplot_kwargs["palette"] = {"F": "xkcd:dark green", "M": "xkcd:dark blue"}
stripplot_kwargs["color"] = "xkcd:dark grey"
legend_kwargs["labels"] = {"F": "Female", "M": "Male"}
legend_kwargs["edgecolor"] = "black"


for idx, rsid in enumerate(rsids_sorted):
    xlabel = f"{GENE_ID} {rsid}"
    columns = [(main_data_type, rsid), (other_data_type, variable)]
    sort_by = [rsid]
    if subgroup is not None:
        columns += [("Metadata", subgroup)]
        sort_by += [subgroup]
    data = df_data.loc[:, columns].droplevel(level=0, axis=1).dropna()
    data = data.convert_dtypes().sort_values(by=sort_by)

    # Create figure and plot
    fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(6, 4))
    ax = plot_violin_for_snp_phenotype(
        data=data,
        rsid=rsid,
        variable=variable,
        subgroup=subgroup,
        split=split,
        ax=ax,
        **dict(
            palette=palette,
            violinplot_kwargs=violinplot_kwargs,
            stripplot_kwargs=stripplot_kwargs,
            boxplot_kwargs=boxplot_kwargs,
            pointplot_kwargs=pointplot_kwargs,
            legend_kwargs=legend_kwargs,
        ),
    )

    ax.grid(True)
    ax.set_xlabel(xlabel, fontdict=fontdict)
    ax.set_ylabel(ylabel, fontdict=fontdict)
    if ax.legend_ and not bool(subgroup):
        ax.legend(legend_labels)

    ax.set_xticks(
        ax.get_xticks(),
        labels=[
            legend_labels.get(float(x.get_text()), x.get_text()).replace(" ", "\n")
            for x in ax.get_xticklabels()
        ],
    )
    ax.set_ylim(ylim)
    ax.xaxis.set_tick_params(labelsize="medium")
    ax.yaxis.set_tick_params(labelsize="medium")
    sns.despine(fig)

    filename_args = [source_type, GENE_ID, df_rsid_map.get(rsid, rsid.replace(":", "."))]
    if subgroup is not None:
        filename_args += [subgroup] + ["split" if split else "seperate"]
    if group_value != "ALL":
        filename_args += [f"{group_key}{group_value}"]
    output_filepath = output_dirpath / make_filename(*filename_args, filetype=filetype)
    fig.savefig(output_filepath)
    logger.info(
        f"Saved '{filetype}' file successfully to '{output_filepath.relative_to(PROJ_ROOT)}'."
    )
    plt.show()
    plt.close()
    summary = pd.concat(
        (
            df_correlations_tall.loc[(variable, rsid)].drop(
                ["Spearman p-value", "Group size (0)", "Group size (1)", "Group size (2)"]
            ),
            data.loc[:, sort_by].value_counts(),
        )
    )
    summary[summary.index.str.contains(" p-value").fillna(False)] = summary[
        summary.index.str.contains(" p-value").fillna(False)
    ].apply(lambda x: "{:.3e}".format(x))
    print(rsid)
    print(summary)
    if n_rsids <= idx:
        break

In [ ]:
subgroup = "Sex"
split = True
rsid = rsids_sorted[0]
pointplot_kwargs = dict(markersize=5 / (1 if not subgroup else 2), c="xkcd:black")
palette = {
    (0.0, "M"): "xkcd:pale blue",
    (1.0, "M"): "xkcd:light blue",
    (2.0, "M"): "xkcd:lightish blue",
    (0.0, "F"): "xkcd:pale green",
    (1.0, "F"): "xkcd:light green",
    (2.0, "F"): "xkcd:lightish green",
}

stripplot_kwargs["palette"] = {"F": "xkcd:dark green", "M": "xkcd:dark blue"}
stripplot_kwargs["color"] = "xkcd:dark grey"
legend_kwargs["labels"] = {"F": "Female", "M": "Male"}
legend_kwargs["edgecolor"] = "black"

xlabel = f"{GENE_ID} {rsid}"
columns = [(main_data_type, rsid), (other_data_type, variable)]
sort_by = [rsid]
if subgroup is not None:
    columns += [("Metadata", subgroup)]
    sort_by += [subgroup]
data = df_data.loc[:, columns].droplevel(level=0, axis=1).dropna()
data = data.convert_dtypes().sort_values(by=sort_by)

# Create figure and plot
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(6, 4))
ax = plot_violin_for_snp_phenotype(
    data=data,
    rsid=rsid,
    variable=variable,
    subgroup=subgroup,
    split=split,
    ax=ax,
    **dict(
        palette=palette,
        violinplot_kwargs=violinplot_kwargs,
        stripplot_kwargs=stripplot_kwargs,
        boxplot_kwargs=boxplot_kwargs,
        pointplot_kwargs=pointplot_kwargs,
        legend_kwargs=legend_kwargs,
    ),
)

ax.grid(True)
ax.set_xlabel(xlabel, fontdict=fontdict)
ax.set_ylabel(ylabel, fontdict=fontdict)
if ax.legend_ and not bool(subgroup):
    ax.legend(legend_labels)

ax.set_xticks(
    ax.get_xticks(),
    labels=[
        legend_labels.get(float(x.get_text()), x.get_text()).replace(" ", "\n")
        for x in ax.get_xticklabels()
    ],
)
ax.set_ylim(ylim)
ax.xaxis.set_tick_params(labelsize="medium")
ax.yaxis.set_tick_params(labelsize="medium")
sns.despine(fig)

filename_args = [source_type, GENE_ID, df_rsid_map.get(rsid, rsid.replace(":", "."))]
if subgroup is not None:
    filename_args += [subgroup] + ["split" if split else "seperate"]
if group_value != "ALL":
    filename_args += [f"{group_key}{group_value}"]
output_filepath = output_dirpath / make_filename(*filename_args, filetype=filetype)
fig.savefig(output_filepath)
logger.info(f"Saved '{filetype}' file successfully to '{output_filepath.relative_to(PROJ_ROOT)}'.")
plt.show()
plt.close()
summary = pd.concat(
    (
        df_correlations_tall.loc[(variable, rsid)].drop(
            ["Spearman p-value", "Group size (0)", "Group size (1)", "Group size (2)"]
        ),
        data.loc[:, sort_by].value_counts(),
    )
)
summary[summary.index.str.contains(" p-value").fillna(False)] = summary[
    summary.index.str.contains(" p-value").fillna(False)
].apply(lambda x: "{:.3e}".format(x))
print(rsid)
print(summary)